# Practical 2a: Deep Feedforward Networks

This notebook implements and analyses deep feedforward networks on the Fashion MNIST dataset, covering:
1.  **Baseline Model**: Training, Validation, and Testing.
2.  **Dropout**: Investigating its effect on overfitting.
3.  **Batch Normalization**: Investigating its effect on convergence and stability.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow executing eagerly: {}".format(tf.executing_eagerly()))

## Data Preparation

In [ ]:
# Load Fashion MNIST dataset
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_and_validation_images, train_and_validation_labels), (test_images, test_labels) = fashion_mnist.load_data()

# Prepare Test Data
test_images_tensor = tf.convert_to_tensor(test_images, dtype=tf.float32) / 255.0
test_labels_tensor = tf.convert_to_tensor(test_labels, dtype=tf.int32)

text_labels = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Construct a validation set from the last 10000 images
validation_images = train_and_validation_images[-10000:, :, :]
validation_labels = train_and_validation_labels[-10000:]

# Construct a training set from the first 50000 images
train_images = train_and_validation_images[:50000, :, :]
train_labels = train_and_validation_labels[:50000]

# Data Pipeline
batch_size = 128
train_ds = tf.data.Dataset.from_tensor_slices((train_images, train_labels))
train_ds = train_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, tf.cast(y, tf.int32)))
train_ds = train_ds.shuffle(buffer_size=batch_size * 10)
train_ds = train_ds.batch(batch_size)

val_ds = tf.data.Dataset.from_tensor_slices((validation_images, validation_labels))
val_ds = val_ds.map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, tf.cast(y, tf.int32)))
val_ds = val_ds.batch(batch_size)

## Model Definition & Experiment Logic

In [ ]:
def create_model(use_dropout=False, use_batchnorm=False):
    """
    Creates a Keras Sequential model with optional Dropout and Batch Normalization.
    """
    layers = [tf.keras.layers.Flatten(input_shape=(28, 28), name='flatten_input')]
    
    # --- Hidden Layer 1 ---
    if use_batchnorm:
        layers.append(tf.keras.layers.Dense(256, use_bias=False, name='input_to_hidden1_linear'))
        layers.append(tf.keras.layers.BatchNormalization(name='hidden1_bn'))
        layers.append(tf.keras.layers.Activation('relu', name='input_to_hidden1_relu'))
    else:
        layers.append(tf.keras.layers.Dense(256, activation='relu', name='input_to_hidden1'))
    
    if use_dropout:
        layers.append(tf.keras.layers.Dropout(0.5, name='hidden1_dropout'))

    # --- Hidden Layer 2 ---
    if use_batchnorm:
        layers.append(tf.keras.layers.Dense(128, use_bias=False, name='hidden1_to_hidden2_linear'))
        layers.append(tf.keras.layers.BatchNormalization(name='hidden2_bn'))
        layers.append(tf.keras.layers.Activation('relu', name='hidden1_to_hidden2_relu'))
    else:
        layers.append(tf.keras.layers.Dense(128, activation='relu', name='hidden1_to_hidden2'))
        
    if use_dropout:
        layers.append(tf.keras.layers.Dropout(0.5, name='hidden2_dropout'))

    # --- Output Layer ---
    layers.append(tf.keras.layers.Dense(10, name='hidden_to_logits'))
    
    return tf.keras.Sequential(layers)


def run_experiment(name, model, train_ds, val_ds, num_epochs=20):
    """
    Runs training and validation for a specific model configuration.
    """
    print(f"\nStarting Experiment: {name}")
    print("-" * 40)
    
    optimizer = tf.keras.optimizers.Adam()
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    
    train_loss = tf.keras.metrics.Mean(name='train_loss')
    train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
    val_loss = tf.keras.metrics.Mean(name='val_loss')
    val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
    
    # Define steps inside to capture specific model/optimizer instances and avoid reusing @tf.function traces
    @tf.function
    def train_step(image, label):
        with tf.GradientTape() as tape:
            logits = model(image, training=True)
            loss = loss_object(label, logits)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        train_loss(loss)
        train_accuracy(label, logits)

    @tf.function
    def val_step(image, label):
        logits = model(image, training=False)
        loss = loss_object(label, logits)
        val_loss(loss)
        val_accuracy(label, logits)
    
    @tf.function
    def test_step(image, label):
        logits = model(image, training=False)
        t_loss = loss_object(label, logits)
        return t_loss, logits

    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'test_loss': None,
        'test_acc': None
    }
    
    for epoch in range(num_epochs):
        train_loss.reset_state()
        train_accuracy.reset_state()
        val_loss.reset_state()
        val_accuracy.reset_state()
        
        for image, label in train_ds:
            train_step(image, label)
            
        for image, label in val_ds:
            val_step(image, label)
            
        print(f'Epoch {epoch + 1}, '
              f'Loss: {train_loss.result():.3f}, Accuracy: {train_accuracy.result():.3%}, '
              f'Val Loss: {val_loss.result():.3f}, Val Accuracy: {val_accuracy.result():.3%}')
        
        history['train_loss'].append(float(train_loss.result()))
        history['train_acc'].append(float(train_accuracy.result()))
        history['val_loss'].append(float(val_loss.result()))
        history['val_acc'].append(float(val_accuracy.result()))

    # Final Test Evaluation
    print(f"Evaluating {name} on Test Set...")
    test_loss_metric = tf.keras.metrics.Mean(name='test_loss')
    test_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='test_accuracy')
    
    t_loss, t_logits = test_step(test_images_tensor, test_labels_tensor)
    test_loss_metric(t_loss)
    test_acc_metric(test_labels_tensor, t_logits)
    
    history['test_loss'] = float(test_loss_metric.result())
    history['test_acc'] = float(test_acc_metric.result())
    print(f"Test Loss: {history['test_loss']:.3f}, Test Accuracy: {history['test_acc']:.3%}")
        
    return history

## 3. Running Experiments

In [ ]:
# 1. Baseline Model
baseline_model = create_model(use_dropout=False, use_batchnorm=False)
baseline_history = run_experiment("Baseline", baseline_model, train_ds, val_ds, num_epochs=20)

# 2. Dropout Model
dropout_model = create_model(use_dropout=True, use_batchnorm=False)
dropout_history = run_experiment("Dropout", dropout_model, train_ds, val_ds, num_epochs=20)

# 3. Batch Normalization Model
batchnorm_model = create_model(use_dropout=False, use_batchnorm=True)
batchnorm_history = run_experiment("Batch Normalization", batchnorm_model, train_ds, val_ds, num_epochs=20)

## 4. Results and Visualization

In [ ]:
epochs_range = range(1, 21)

# Plot 1: Compare Validation Accuracy
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, baseline_history['val_acc'], label='Baseline Val Acc')
plt.plot(epochs_range, dropout_history['val_acc'], label='Dropout Val Acc')
plt.plot(epochs_range, batchnorm_history['val_acc'], label='BatchNorm Val Acc')
plt.title('Validation Accuracy Comparison')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.savefig('comparison_accuracy.png')
plt.show()

# Plot 2: Compare Validation Loss
plt.figure(figsize=(10, 6))
plt.plot(epochs_range, baseline_history['val_loss'], label='Baseline Val Loss')
plt.plot(epochs_range, dropout_history['val_loss'], label='Dropout Val Loss')
plt.plot(epochs_range, batchnorm_history['val_loss'], label='BatchNorm Val Loss')
plt.title('Validation Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.savefig('comparison_loss.png')
plt.show()

# Plot 3: Detailed Training vs Validation for Dropout
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, dropout_history['train_acc'], label='Train Acc')
plt.plot(epochs_range, dropout_history['val_acc'], label='Val Acc')
plt.title('Dropout: Training vs Validation Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, dropout_history['train_loss'], label='Train Loss')
plt.plot(epochs_range, dropout_history['val_loss'], label='Val Loss')
plt.title('Dropout: Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig('dropout_performance.png')
plt.show()

# Plot 4: Question 1 specific - Baseline Training vs Validation
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, baseline_history['train_acc'], label='Train Acc')
plt.plot(epochs_range, baseline_history['val_acc'], label='Val Acc')
plt.title('Baseline: Training vs Validation Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs_range, baseline_history['train_loss'], label='Train Loss')
plt.plot(epochs_range, baseline_history['val_loss'], label='Val Loss')
plt.title('Baseline: Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.savefig('baseline_performance.png')
plt.show()